# Assignment 0 (Week 1): Flight-log quality check
# تمرین صفر (هفته‌ی ۱): کنترل کیفیت لاگ پرواز

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0 &nbsp;|&nbsp;
**Released:** Week 1 &nbsp;|&nbsp; **Due:** start of Week 2 &nbsp;|&nbsp; **Effort:** about 3 to 4 hours

> **Provenance:** new content. Pandas/Matplotlib usage follows the
> [W3Schools Pandas tutorial](https://www.w3schools.com/python/pandas/default.asp) and the
> [GeeksforGeeks describe() article](https://www.geeksforgeeks.org/python/python-pandas-dataframe-describe-method/);
> no text is copied from either.

> Numbering note: this warm-up is **Assignment 0** so that the graded milestones keep the
> syllabus numbers (Assignment 1 = Week 2 Sensor OOP hierarchy).

<div dir="rtl">

## صورت مسئله

یک پهپاد بال‌ثابت آزمایشی پرواز پنج‌دقیقه‌ای دیگری انجام داده است: برخاست، صعود، یک دور تند، یک مانور بالاکشیدن (pull-up)، کاهش ارتفاع و فرود. داده‌های IMU با نرخ ۵۰ هرتز همراه با ارتفاع بارومتری، سرعت هوایی و دمای هوا در فایل
<code>data/sample/assignment00_uav_flight_log.csv</code>
ذخیره شده است.

حد سازه‌ای بدنه **۲٫۰g** است. اگر ضریب بار واقعی از این حد عبور کرده باشد، هواپیما باید برای بازرسی زمین‌گیر شود. اما لاگر عیوبی دارد: چند نمونه‌ی خراب (خار)، مقادیر گمشده و یک قطعی در ثبت داده.

وظیفه‌ی شما نوشتن توابعی تمیز، با نوع‌دهی (type hints) و مستند است که این داده را بررسی کند و در پایان یک حکم مهندسی صادر کنید.

**قواعد:**
- فقط در سلول‌هایی که با <code># TODO</code> مشخص شده‌اند کد بنویسید؛ امضای توابع را تغییر ندهید.
- از حلقه‌ی پایتون روی سطرهای داده استفاده نکنید (عملیات برداری NumPy/Pandas).
- سلول‌های «HIDDEN TESTS» را حذف یا ویرایش نکنید؛ مدرس هنگام نمره‌دهی آن‌ها را جایگزین می‌کند.
- همه‌ی توضیحات داخل کد (comments) به انگلیسی باشد.
- پیش از تحویل: Kernel → Restart & Run All. نوت‌بوک باید بدون خطا تا انتها اجرا شود.

</div>

## Problem statement (English summary)

Another 5-minute UAV test flight (take-off, climb, steep turn, pull-up, descent, landing) was
logged at 50 Hz in `data/sample/assignment00_uav_flight_log.csv`. The airframe limit is
**2.0 g**; an exceedance grounds the aircraft. The logger has defects (spikes, NaN dropouts,
an outage). Write clean, typed, documented functions that check the log, then give an
engineering verdict.

**Rules:** write code only in `# TODO` cells and keep the function signatures; no Python
loops over rows; do not delete or edit the `HIDDEN TESTS` cells; English-only code comments;
before submitting, *Restart & Run All* must finish without errors.

## Setup (run first, do not edit)

In [ ]:
from __future__ import annotations

import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.figure import Figure


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``data/sample``."""
    for folder in (start, *start.parents):
        if (folder / "data" / "sample").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
DATA_PATH: Final[Path] = REPO_ROOT / "data" / "sample" / "assignment00_uav_flight_log.csv"

G0: Final[float] = 9.80665
LOAD_LIMIT_G: Final[float] = 2.0
GROUND_END_S: Final[float] = 25.0
GAP_TOLERANCE: Final[float] = 1.5
MEDIAN_WINDOW: Final[int] = 5
TIME_COLUMN: Final[str] = "time_s"
ACCEL_COLUMNS: Final[list[str]] = ["accel_x_mps2", "accel_y_mps2", "accel_z_mps2"]
GYRO_COLUMNS: Final[list[str]] = ["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]
REQUIRED_COLUMNS: Final[list[str]] = [
    TIME_COLUMN, *ACCEL_COLUMNS, *GYRO_COLUMNS, "baro_alt_m", "airspeed_mps", "temp_c",
]

## Tasks

### Task 1 (5 pts): environment report
<div dir="rtl">تابعی بنویسید که نسخه‌ی پایتون و سه کتابخانه‌ی numpy، pandas و matplotlib را به صورت یک دیکشنری برگرداند.</div>

Return a dictionary with keys `"python"`, `"numpy"`, `"pandas"`, `"matplotlib"` mapping to
version strings such as `"3.11.8"`.

In [ ]:
def environment_report() -> dict[str, str]:
    """Return the Python version and the versions of numpy, pandas and matplotlib."""
    # TODO: use sys.version_info (or platform) and importlib.metadata.version
    raise NotImplementedError

In [ ]:
# HIDDEN TESTS (task1): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 2 (10 pts): load and validate the log
<div dir="rtl">فایل CSV را بخوانید. اگر هر یک از ستون‌های لازم وجود نداشت، خطای ValueError با نام ستون‌های گمشده بدهید. داده را بر حسب زمان مرتب کنید و اندیس را از نو بسازید.</div>

* Read the CSV with Pandas.
* If any column in `REQUIRED_COLUMNS` is missing, raise `ValueError` whose message names
  the missing columns.
* Sort by `time_s` and reset the index (0, 1, 2, ...).

In [ ]:
def load_flight_log(path: Path) -> pd.DataFrame:
    """Load a flight-log CSV, validate its columns and sort it by time.

    Args:
        path: CSV file to read.

    Returns:
        DataFrame sorted by ``time_s`` with a fresh RangeIndex.

    Raises:
        ValueError: If a required column is missing.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# Public test (visible): runs on the real file.
try:
    log = load_flight_log(DATA_PATH)
    assert list(log.columns) == REQUIRED_COLUMNS
    assert log[TIME_COLUMN].is_monotonic_increasing
    print("[x] Task 2 public test passed:", log.shape)
except NotImplementedError:
    log = pd.read_csv(DATA_PATH)   # fallback so later cells still run
    print("[ ] Task 2 not done yet (using a fallback loader)")

In [ ]:
# HIDDEN TESTS (task2): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 3 (15 pts): sample rate and gaps
<div dir="rtl">نرخ نمونه‌برداری را با میانه‌ی اختلاف زمانی تخمین بزنید. سپس همه‌ی شکاف‌ها (گام‌های بزرگ‌تر از ۱٫۵ برابر گام اسمی) را با زمان شروع، پایان و تعداد نمونه‌های گمشده گزارش کنید.</div>

* `estimate_sample_rate(time_s)` returns $1/\operatorname{median}(\Delta t)$ in Hz.
* `find_gaps(time_s, tolerance)` returns a DataFrame with columns `gap_start_s`,
  `gap_end_s`, `missing_samples` (int), one row per step longer than
  `tolerance` × nominal step, with a fresh RangeIndex. Return an **empty** DataFrame with the
  same columns when there is no gap.

In [ ]:
def estimate_sample_rate(time_s: pd.Series) -> float:
    """Return the sampling rate in Hz from the median time step."""
    # TODO
    raise NotImplementedError


def find_gaps(time_s: pd.Series, tolerance: float = GAP_TOLERANCE) -> pd.DataFrame:
    """Return one row per logger gap: gap_start_s, gap_end_s, missing_samples."""
    # TODO
    raise NotImplementedError

In [ ]:
# Public test (visible): the worked example from the lecture.
try:
    t = pd.Series([99.94, 99.96, 99.98, 101.20, 101.22])
    assert np.isclose(estimate_sample_rate(t), 50.0)
    g = find_gaps(t)
    assert list(g.columns) == ["gap_start_s", "gap_end_s", "missing_samples"]
    assert g["missing_samples"].tolist() == [60]
    print("[x] Task 3 public test passed")
except NotImplementedError:
    print("[ ] Task 3 not done yet")

In [ ]:
# HIDDEN TESTS (task3): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 4 (10 pts): missing-value report
<div dir="rtl">برای هر ستون، تعداد و درصد مقادیر گمشده (NaN) را در یک DataFrame گزارش کنید، به ترتیب نزولی درصد.</div>

Return a DataFrame indexed by column name with columns `n_missing` (int) and
`pct_missing` (float, 0 to 100), sorted by `pct_missing` descending.

In [ ]:
def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return n_missing and pct_missing per column, most-missing first."""
    # TODO
    raise NotImplementedError

In [ ]:
# HIDDEN TESTS (task4): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 5 (20 pts): load factor, spikes and exceedances
<div dir="rtl">ضریب بار را از شتاب‌سنج حساب کنید، با میانه‌ی متحرک (پنجره‌ی ۵) خارهای تک‌نمونه‌ای را حذف کنید، و همه‌ی بازه‌های پیوسته‌ای را که ضریب بار فیلترشده از حد مجاز بیشتر است با زمان شروع، پایان، مدت و بیشینه گزارش کنید.</div>

* `load_factor(df)`: $n = \lVert \mathbf f \rVert / g_0$ as a Series (vectorised, no loop).
* `filtered_load_factor(n, window)`: centred rolling median, `min_periods=1`.
* `exceedances(n, time_s, limit)`: one row per **contiguous** run of samples with
  `n > limit`; columns `start_s`, `end_s`, `duration_s`, `peak_n`. Define
  `duration_s` as (number of samples in the run) × (nominal time step). Return an empty
  DataFrame with these columns if there is none.

Hint for runs: `run_id = (mask != mask.shift()).cumsum()` labels each run of equal values.

In [ ]:
def load_factor(df: pd.DataFrame) -> pd.Series:
    """Return n = |accel| / g0 for every row."""
    # TODO
    raise NotImplementedError


def filtered_load_factor(n: pd.Series, window: int = MEDIAN_WINDOW) -> pd.Series:
    """Return the centred rolling median of the load factor."""
    # TODO
    raise NotImplementedError


def exceedances(n: pd.Series, time_s: pd.Series, limit: float = LOAD_LIMIT_G) -> pd.DataFrame:
    """Return contiguous intervals where n > limit: start_s, end_s, duration_s, peak_n."""
    # TODO
    raise NotImplementedError

In [ ]:
# Public test (visible): a hand-made signal with one spike and one real excursion.
try:
    t = pd.Series(np.arange(12) * 0.02)
    n = pd.Series([1.0, 1.0, 5.0, 1.0, 1.0, 2.2, 2.3, 2.4, 2.3, 2.2, 1.0, 1.0])
    n_f = filtered_load_factor(n)
    events = exceedances(n_f, t)
    assert len(events) == 1                                # the spike is gone
    assert np.isclose(events.loc[0, "peak_n"], 2.3)
    print("[x] Task 5 public test passed")
    print(events)
except NotImplementedError:
    print("[ ] Task 5 not done yet")

In [ ]:
# HIDDEN TESTS (task5): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 6 (10 pts): gyro noise at rest
<div dir="rtl">در ۲۵ ثانیه‌ی نخست هواپیما ساکن است. بایاس (میانگین) و انحراف معیار نمونه‌ای (ddof=1) هر سه ژیروسکوپ را گزارش کنید.</div>

Return a DataFrame indexed by gyro column name with columns `bias_dps` and `std_dps`
(sample standard deviation, `ddof=1`) computed over `time_s < ground_end_s`.

In [ ]:
def gyro_noise(df: pd.DataFrame, ground_end_s: float = GROUND_END_S) -> pd.DataFrame:
    """Return bias_dps and std_dps of each gyro while the aircraft is parked."""
    # TODO
    raise NotImplementedError

In [ ]:
# HIDDEN TESTS (task6): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 7 (15 pts): the chief engineer's dashboard
<div dir="rtl">یک شکل چهار پنلی با محور زمان مشترک بسازید: ارتفاع، سرعت هوایی، ضریب بار خام و فیلترشده همراه با خط حد ۲g، و سرعت زاویه‌ای سه ژیروسکوپ. شکاف‌های لاگر را با نوار خاکستری نشان دهید. تابع باید شیء Figure را برگرداند.</div>

Build a 4-row figure with a shared time axis:
1. barometric altitude; 2. airspeed; 3. raw **and** filtered load factor with a dashed
2.0 g line; 4. the three gyro rates. Shade every logger gap (from `find_gaps`) in all panels.
Label every axis with units. **Return** the `Figure`; do not call `plt.show()` inside.

In [ ]:
def plot_dashboard(df: pd.DataFrame) -> Figure:
    """Return a 4-panel overview figure of the flight log."""
    # TODO
    raise NotImplementedError

In [ ]:
try:
    fig = plot_dashboard(log)
    assert isinstance(fig, Figure) and len(fig.axes) >= 4
    plt.show()
    print("[x] Task 7 public test passed")
except NotImplementedError:
    print("[ ] Task 7 not done yet")

In [ ]:
# HIDDEN TESTS (task7): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 8 (15 pts): engineering verdict and Git history
<div dir="rtl">با استفاده از خروجی توابع بالا، در حداکثر ۲۰۰ کلمه (فارسی یا انگلیسی) به این پرسش پاسخ دهید: «آیا این هواپیما باید برای بازرسی سازه زمین‌گیر شود؟» پاسخ باید شامل عدد بیشینه‌ی ضریب بار فیلترشده، زمان و مدت هر عبور از حد، و دلیل نادیده گرفتن خارهای خام باشد. سپس خروجی دستور <code>git log --oneline</code> با دست‌کم سه کامیت از کار خودتان را در سلول بعدی قرار دهید.</div>

In at most 200 words (Persian or English), answer: **should this aircraft be grounded for a
structural inspection?** Quote the peak filtered load factor, the time and duration of each
exceedance, and why the raw peaks were dismissed. Then paste the output of
`git log --oneline` showing at least three commits of your own work.

In [ ]:
# Run this cell to print the numbers your verdict should quote.
try:
    n_raw = load_factor(log)
    n_filtered = filtered_load_factor(n_raw)
    print(f"raw peak      : {n_raw.max():.2f} g at t = {log.loc[n_raw.idxmax(), TIME_COLUMN]:.2f} s")
    print(f"filtered peak : {n_filtered.max():.2f} g at t = {log.loc[n_filtered.idxmax(), TIME_COLUMN]:.2f} s")
    print(exceedances(n_filtered, log[TIME_COLUMN]))
except NotImplementedError:
    print("[ ] Task 5 must be done first")

**Your verdict (≤ 200 words):**

_Write here._

**`git log --oneline`:**

```
paste here
```

## Expected output (check yourself against these)

| Quantity | Expected value |
|----------|----------------|
| Shape after loading | (14 900, 10) |
| Sample rate | 50.0 Hz |
| Logger gaps | 1 gap: 174.98 s → 177.00 s, 100 missing samples |
| Missing values | `baro_alt_m` and `airspeed_mps`: 149 each (1.00 %), other columns 0 |
| Raw peak load factor | ≈ 5.47 g (a single-sample spike) |
| Filtered peak load factor | ≈ 2.25 g at t ≈ 202.4 s |
| Exceedances of 2.0 g (filtered) | 1 interval, ≈ 201.8 s → 203.2 s, ≈ 1.46 s, peak ≈ 2.25 g |
| Gyro noise at rest | bias ≈ 0 deg/s, std ≈ 0.30 deg/s on each axis |

Your numbers may differ in the last digit; the hidden tests use tolerances.